# CFNA one-click Colab/Jupyter setup, corpus build, training, and inference

Run cells from top to bottom. The first cell is intentionally self-contained for Google Colab: if the notebook is not already inside the repository, it clones the repo into `/content/nueronce`, installs dependencies, installs `cfna` editable, and changes the working directory to the repo root.

**Corpus policy:** the build uses only the sources registered in `cfna.corpus.sources.safe_commercial_sources()` and the corpus builder enforces allowed source hosts plus commercial-safe public-domain/CC0 license IDs. No blogs, social media, forums, scraped web, or copyrighted sources are used. The notebook also validates the manifest before training, so it will fail rather than train on an unapproved source/license.


In [ ]:
# 1) READY-TO-RUN SETUP FOR COLAB/JUPYTER
# If you forked the repo, change REPO_URL. Otherwise run this cell as-is.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/lmminier/nueronce.git"
BRANCH = None  # Example: "main". Leave as None to use the repo default branch.
REPO_DIR = Path("/content/nueronce") if Path("/content").exists() else Path.cwd()

# If this notebook is opened by itself in Colab/Drive, clone the code first.
if not (REPO_DIR / "pyproject.toml").exists():
    clone_cmd = ["git", "clone", "--depth", "1"]
    if BRANCH:
        clone_cmd += ["--branch", BRANCH]
    clone_cmd += [REPO_URL, str(REPO_DIR)]
    subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def run_step(cmd):
    """Run a script and always print its stdout/stderr before raising, so a
    failure never shows up as a bare CalledProcessError with no visible cause
    (capture_output alone would otherwise hide the real error)."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print("--- stdout ---")
        print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    result.check_returncode()
    return result


# Colab normally already has numpy + torch, but requirements keeps local/Jupyter installs consistent.
run_step([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run_step([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-build-isolation"])

print("Ready. Working directory:", Path.cwd())

In [ ]:
# 2) Verify imports and hardware.
from pathlib import Path
import sys

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch
import cfna

print("repo:", REPO)
print("cfna:", cfna.__version__)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# 3) Inspect the only sources this notebook will use.
from cfna.corpus.sources import safe_commercial_sources, EXCLUDED_KINDS

sources = safe_commercial_sources()
for source in sources:
    print(f"{source.source_id}: {source.url} | {source.license_id} | bucket={source.bucket}")

print("\nExplicitly excluded source kinds:", ", ".join(EXCLUDED_KINDS))


In [ ]:
# 4) Build the allowed public-domain corpus.
# Downloads curated NLTK-hosted Project Gutenberg + US-government speech archives
# via raw.githubusercontent.com, then writes <CORPUS_DIR>/manifest.jsonl.
#
# CORPUS_DIR is defined once here and reused by cells 5 and 6 below, so the
# build step and the training step can never point at two different
# directories (that mismatch is the #1 cause of "corpus not found" errors).
CORPUS_DIR = "corpus"

run_step([sys.executable, "scripts/build_corpus.py", "--out", CORPUS_DIR])

In [ ]:
# 5) Verify every manifest record is allowed, and that there is actually
# enough data to train on — fail fast here with a clear message rather than
# deep inside cell 6.
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

manifest_path = Path(CORPUS_DIR) / "manifest.jsonl"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"{manifest_path} does not exist. Run cell 4 (build the corpus) first, "
        f"and make sure CORPUS_DIR matches what it was built with (currently "
        f"CORPUS_DIR={CORPUS_DIR!r})."
    )

records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
allowed_hosts = {"raw.githubusercontent.com"}
allowed_license_ids = {"public-domain-us", "public-domain-usgov", "CC0-1.0"}

assert records, "corpus manifest is empty"
for record in records:
    assert record["bucket"] == "safe_commercial", record["document_id"]
    assert record["commercial_use"] is True, record["document_id"]
    assert record["attribution_required"] is False, record["document_id"]
    assert record["license_id"] in allowed_license_ids, record["document_id"]
    assert urlparse(record["source_locator"]).hostname in allowed_hosts, record["document_id"]

by_split = Counter(r["split"] for r in records)
if len(records) < 2 or by_split["train"] == 0 or by_split["val"] == 0:
    raise ValueError(
        f"The corpus needs at least one training document and one validation "
        f"document; got {dict(by_split)}. This usually means a source failed "
        f"to download (scroll up to cell 4's output) or the source list no "
        f"longer matches scripts/build_corpus.py's VAL_DOCS. Re-run cell 4 "
        f"and check network access to raw.githubusercontent.com if it persists."
    )

print(f"documents: {len(records)}")
print("splits:", dict(by_split))
print("licenses:", dict(Counter(r["license_id"] for r in records)))
print("bytes:", sum(r["n_bytes"] for r in records))

In [ ]:
# 6) Train a checkpoint.
# For a quick smoke test, use 0.25-1 minute. For better generations, use 20+ minutes.
TRAIN_MINUTES = 1.0
CKPT = "checkpoints/cfna_chat.pt"

run_step([
    sys.executable, "scripts/train_checkpoint.py",
    "--corpus", CORPUS_DIR,
    "--minutes", str(TRAIN_MINUTES),
    "--out", CKPT,
])

In [ ]:
# 7) Run the built-in chat/inference demo.
run_step([
    sys.executable, "scripts/chat_demo.py",
    "--ckpt", CKPT,
    "--temp", "0.7",
    "--max-new", "120",
    "--seed", "0",
])

In [ ]:
# 8) Ask your own prompt from inside the notebook.
from cfna.chat import Conversation, load_checkpoint

model, ckpt = load_checkpoint(CKPT)
conversation = Conversation(model, system="A conversation.", temperature=0.7, max_new=120)

prompt = "Hello CFNA. Can you describe liberty in one paragraph?"
reply = conversation.say(prompt)
print("User:", prompt)
print("Assistant:", reply)
